Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS regularization wave2

35 variants,2 seeds,11 outer folds (770 fits). Simple changes first,then six fixed combinations. Label smoothing;input/feature augmentations;window/EMA/SWAD averaging.

**Unchanged:** inner cell-NLL epoch selection,Shallow16,class-balanced loss,AdamW.001,WD.01,dropout.5,causal alignment prior5. No pretrained models.

Only regularizers run;no separate controls. Historical results are reused for comparison. Clean held-out evaluation;training-only BN recalibration. Two T4workers,240minute wall cap;partial results labeled. No competition submission.

In [ ]:
import os,sys,pathlib,json,hashlib,zipfile,io,base64,subprocess
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/reg-mpl")
import torch
assert torch.cuda.is_available(),"Enable requested Kaggle GPU"
print("GPU",torch.cuda.get_device_name(0),flush=True)
project=pathlib.Path("/kaggle/working/riemann-project");project.mkdir(exist_ok=True)
sources=list(pathlib.Path("/kaggle/input").rglob("source-integrity.json"));assert len(sources)==1
raw=sources[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(raw).items():
    p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
    data=(sources[0].parent/rel).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
payload=base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text())
assert hashlib.sha256(payload).hexdigest()=="3b3615fc6f6de5f6824fd234e0b6fc85620e52c2bc95aa44b97b359453c9c6b9"
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read("regularization-manifest.json"))
    for rel,digest in manifest.items():
        p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
        data=z.read(rel);assert hashlib.sha256(data).hexdigest()==digest
        dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
repro=project/"reproduction";experiment=repro/"regularization_research_20261108"
inputs=list(pathlib.Path("/kaggle/input").rglob("input-integrity.json"));assert len(inputs)==1
input_manifest=json.loads(inputs[0].read_text());input_dir=inputs[0].parent
for name,digest in input_manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (input_dir/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    (repro/"data"/f"proteus_{split}.npz").symlink_to(input_dir/f"proteus_{split}.npz")
(repro/"temporal_spd_research/cache").symlink_to(input_dir,target_is_directory=True)
envdir=pathlib.Path("/tmp/reg-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--quiet",*pins],check=True)
out=pathlib.Path("/kaggle/working/regularization-results");out.mkdir(exist_ok=True)
env=dict(os.environ,REG_OUTPUT=str(out),REG_CACHE="/tmp/proteus-regularization-cache")
(out/"bundle-provenance.json").write_text(json.dumps(dict(regularization_bundle_sha256="3b3615fc6f6de5f6824fd234e0b6fc85620e52c2bc95aa44b97b359453c9c6b9",original_source_manifest_sha256=hashlib.sha256(raw).hexdigest(),input_manifest=input_manifest,source_manifest=manifest),indent=2))
print("Verified public data and author ShallowNet. All10people,35regularizers,unchanged NLL stopping,240min cap.",flush=True)

print("CUDA DEVICES",torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)


## CPU regression tests and all35 CUDA gradient smokes

In [ ]:
subprocess.run([python,"-m","pytest","-q","tests/test_regularizers.py","tests/test_training.py","tests/test_pipeline.py","--tb=short"],cwd=experiment,env=env,check=True)
subprocess.run([python,"-u","smoke.py","--device","cuda:0"],cwd=experiment,env=env,check=True)


## Simple-first grid;NLL selection and fresh outer refit

In [ ]:
process=subprocess.Popen([python,"-u","run_gpu.py","--minutes","240"],cwd=experiment,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end="",flush=True)
code=process.wait()
if code:raise RuntimeError(f"Regularization study exited{code};partial checkpoints preserved")


## Complete two-seed configurations only

In [ ]:
print((out/"REPORT.md").read_text())
print("Learning curves:",len(list(out.rglob("curves.png"))))